# Model A – Standard CNN (EuroSAT, 10 land-use classes)

EN3150 Assignment 03 – custom CNN that interleaves **standard 2D convolutions** and **max-pooling**.

**Data:** the shared EuroSAT split built by `Dataset.ipynb` in `data/eurosat/train|val|test/<class>/`
(64×64 RGB satellite images, 18,900 / 4,050 / 4,050). Run `Dataset.ipynb` once before this notebook.
No data augmentation.

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR    = Path("data/eurosat")   # built by Dataset.ipynb
RESULTS_DIR = Path("results")        # numbers and saved model (format: results/README.md)
FIGURES_DIR = Path("figures")        # figures for the report
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

IMG_SIZE = (64, 64)
SEED     = 42
BATCH    = 32
AUTOTUNE = tf.data.AUTOTUNE

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none")

TensorFlow 2.21.0 | GPU: none


In [2]:
def load_split(name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        str(DATA_DIR / name), image_size=IMG_SIZE, batch_size=BATCH,
        shuffle=shuffle, seed=SEED, label_mode="int")

train_raw = load_split("train", shuffle=True)
val_raw   = load_split("val",   shuffle=False)
test_raw  = load_split("test",  shuffle=False)

CLASS_NAMES = train_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(NUM_CLASSES, "classes:", CLASS_NAMES)

# Scale pixels from 0-255 to 0-1. Keep the decoded images in memory (cache) and
# prepare the next batch while the current one trains (prefetch).
def scale(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

train_ds = train_raw.map(scale, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds   = val_raw.map(scale,   num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds  = test_raw.map(scale,  num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

Found 18900 files belonging to 10 classes.


Found 4050 files belonging to 10 classes.


Found 4050 files belonging to 10 classes.


10 classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## 1. Architecture

Four blocks of **Conv2D (3×3) → BatchNorm → ReLU → MaxPooling (2×2)** with filters doubling
32 → 64 → 128 → 256, then **Global Average Pooling → Dense(128, ReLU) → Dropout(0.5) → Dense(10, softmax)**.

| Layer | Output shape | Kernel / units | Parameters |
|---|---|---|---|
| Input | 64×64×3 | – | 0 |
| Conv2D (no bias) | 64×64×32 | 3×3, 32 filters | 3·3·3·32 = 864 |
| BatchNorm + ReLU | 64×64×32 | – | 4·32 = 128 (64 trainable) |
| MaxPool | 32×32×32 | 2×2 | 0 |
| Conv2D (no bias) | 32×32×64 | 3×3, 64 filters | 3·3·32·64 = 18,432 |
| BatchNorm + ReLU | 32×32×64 | – | 4·64 = 256 (128 trainable) |
| MaxPool | 16×16×64 | 2×2 | 0 |
| Conv2D (no bias) | 16×16×128 | 3×3, 128 filters | 3·3·64·128 = 73,728 |
| BatchNorm + ReLU | 16×16×128 | – | 4·128 = 512 (256 trainable) |
| MaxPool | 8×8×128 | 2×2 | 0 |
| Conv2D (no bias) | 8×8×256 | 3×3, 256 filters | 3·3·128·256 = 294,912 |
| BatchNorm + ReLU | 8×8×256 | – | 4·256 = 1,024 (512 trainable) |
| MaxPool | 4×4×256 | 2×2 | 0 |
| GlobalAvgPool | 256 | – | 0 |
| Dense + ReLU | 128 | 128 units | 256·128 + 128 = 32,896 |
| Dropout | 128 | rate 0.5 | 0 |
| Dense + Softmax | 10 | 10 units | 128·10 + 10 = 1,290 |
| **Total** | | | **424,042** (423,082 trainable + 960 BatchNorm statistics) |

Conv params = K·K·C_in·C_out (no bias – BatchNorm adds its own shift). BatchNorm has 4 values per channel:
γ and β are trained, the running mean and variance are statistics. Dense params = inputs·units + units.

Batch normalisation was added after comparing variants on validation data: it raised validation accuracy by
about 4 percentage points at the same size, and it costs nothing at inference because it can be folded
into the preceding convolution.

In [3]:
def build_model_a(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES):
    """Model A – standard CNN. Calling it again gives a model with fresh weights."""
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, kernel_size=3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs, name="Model_A_StandardCNN")

model_a = build_model_a()
model_a.summary()

Model: "Model_A_StandardCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 256)      │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 424,042 (1.62 MB)

 Trainable params: 423,082 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [4]:
# Check: a batch of 64x64x3 images gives one probability vector of length 10 per image
x_batch, y_batch = next(iter(val_ds))
probs = model_a.predict(x_batch, verbose=0)
n_trainable = int(sum(np.prod(w.shape) for w in model_a.trainable_weights))
print("pixel range       :", float(x_batch.numpy().min()), "to", float(x_batch.numpy().max()))
print("input  shape      :", x_batch.shape)
print("output shape      :", probs.shape)
print("rows sum to 1     :", bool(np.allclose(probs.sum(axis=1), 1.0, atol=1e-5)))
print("total params      :", f"{model_a.count_params():,}")
print("trainable params  :", f"{n_trainable:,}")
assert probs.shape == (len(x_batch), NUM_CLASSES)

pixel range       : 0.09803921729326248 to 1.0
input  shape      : (32, 64, 64, 3)
output shape      : (32, 10)
rows sum to 1     : True
total params      : 424,042
trainable params  : 423,082
